pydantic base model ()

class Prompter:

    def __init__(self, db_path: str, config:Dict [str]): (dbpath
        keep dbpath if end with .tmp else make dbpath with .tmp ending
        config = config
        create duckdb con
        make sure columns have right columns like status etc

    __exit__


    setup db
        check status column

    async send single prompt
        make sure status is not complete

    resume/start loop
        start at row with no status and gather them into transactions to do
        -- use ACID for sql transactions 
            Atomicity: A transaction is either fully completed or fully rolled back. If any part fails, the entire transaction fails.
            Consistency: A transaction moves the database from one valid state to another while following all rules and constraint.
            Isolation: Transactions run independently of each other, even when executed at the same time.
            Durability: Once a transaction is committed, its changes remain permanent, even after a system failure

    finalize
        change db file name from .duckdb.tmp -> .duckdb once all status's are complete
        


    
    

In [4]:
from threading import Thread, current_thread
import duckdb
import requests
from datetime import timedelta
from pydantic import BaseModel, ValidationError, BeforeValidator, field_validator
from typing import Optional, List, Dict, Any
from pathlib import Path
from enum import StrEnum
import re

# https://duckdb.org/docs/stable/guides/python/multiple_threads
# https://duckdb.org/docs/stable/sql/statements/transactions
# https://en.wikipedia.org/wiki/Database_transaction

# https://pmc.ncbi.nlm.nih.gov/articles/PMC9081216/

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL = "tinyllama"
DATA_PATH = "../data/raw/example_variant_data.csv"

prompt = """You are an expert system.
    Analyze the following somatic variant data and predict its pathogenicity
    (do not rely on any previous examples or context):
    
    ### INPUT DATA
    Chromosome: 3
    ChromosomePosition: 178937422
    Ref: T
    RefAA: Cys
    Alt: C
    AltAA: Arg
    Type: missense
    Coverage: 806
    Gene: PIK3CA
    GeneStrand: +
    ExonNumber: 12
    Transcript: NM_006218.2
    Protein: NP_006209.2
    CodingBase: 1810
    CodonPosition: 1
    AAPosition: 604
    HGVSGenomic: g.178937422T>C
    HGVSCodingTranscript: NM_006218.2
    HGVSCoding: c.1810T>C
    HGVSTranslationProtein: NP_006209.2
    HGVSProtein: p.Cys604Arg
    ### TASK
    Based on this information, please classify the variant as 'Benign', 'Likely Benign', 'Likely Pathogenic',
    'Pathogenic', or 'Unknown Significance'. Answer with only one type."""

prompt_test = 'Hello!'


class Classification(StrEnum):
    BENIGN = 'Benign'
    LIKELY_BENIGN = 'Likely Benign'
    PATHOGENIC = 'Pathogenic'
    LIKELY_PATHOGENIC = 'Likely Pathogenic'
    VUS = 'VUS'

class Pathogenicity(BaseModel):
    classification: Classification

    @field_validator('classification', mode='before')
    @classmethod
    def parse(cls, value: str | Classification) -> Classification:
        if isinstance(value, Classification):
            return value

        text = re.sub(r'[^a-z0-9]', ' ', str(value).lower()).strip()

        mapping = {
            # '[ \-_]*' means zero or more spaces, dashes, or underscores
            r'variant[ \-_]*of[ \-_]*uncertain[ \-_]*significance': Classification.VUS,
            r'variant[ \-_]*of[ \-_]*unknown[ \-_]*significance': Classification.VUS,
            r'uncertain[ \-_]*significance': Classification.VUS,
            r'unknown[ \-_]*significance': Classification.VUS,
            
            r'likely[ \-_]*pathogenic': Classification.LIKELY_PATHOGENIC,
            r'likely[ \-_]*oncogenic': Classification.LIKELY_PATHOGENIC,
            
            r'likely[ \-_]*benign': Classification.LIKELY_BENIGN,
            
            # No regex needed for single words
            r'pathogenic': Classification.PATHOGENIC,
            r'oncogenic': Classification.PATHOGENIC,
            r'benign': Classification.BENIGN,
            r'vus': Classification.VUS,
        }

        found = set()

        sorted_patterns = sorted(mapping.keys(), key=len, reverse=True)
        
        for pattern in  sorted_patterns:
            regex = re.compile(rf'\b{pattern}\b')
            if regex.search(text):
                found.add(mapping[pattern])
                text = regex.sub('', text)

        print(found)
        
        if len(found) == 1:
            return list(found)[0]
        elif len(found) > 1:
            raise ValueError(f"Multiple classifications detected: {found} '{value}'")
        else:    
            raise ValueError(f"Could not map '{value}' to a valid Classification")

class LLMResponse(BaseModel):
    response: dict
    status_code: int
    pathogenicity: Pathogenicity
    time_elapsed: timedelta

class Prompter:
    """
    This is a reST style.
    
    :param param1: this is a first param
    :param param2: this is a second param
    :returns: this is a description of what is returned
    :raises keyError: raises an exception
    """


    def __init__(self, db_path: str='test', config: Path = 'test'):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        self.db_path = db_path #if db_path.endswith(".tmp") else f"{db_path}.tmp"
        self.config = config
        self.con = duckdb.connect(self.db_path)
        self._setup_db()

    def _setup_db(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # SQL logic to ensure 'status' (pending/completed/failed) columns exist
        self.con.execute("""ALTER TABLE prompts ADD COLUMN IF NOT EXISTS status STRING DEFAULT "PENDING";""")
        self.con.execute("ALTER TABLE prompts ADD COLUMN IF NOT EXISTS response STRING DEFAULT NULL")
        self.con.execute("ALTER TABLE prompts ADD COLUMN IF NOT EXISTS prediction STRING DEFAULT NULL")
        self.con.sql("SELECT * FROM prompts").show()
        pass

    def send_single_prompt(self, row_id: str, prompt: str) -> LLMResponse:
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception

        make new duckdb.con for each prompt
        """
        try:
            # Async HTTP request logic here
            # response = await client.post(...)
            
            # Runtime Validation
            # return LLMResponse.model_validate(response.json())
            response = requests.post(
                OLLAMA_URL,
                json={
                    "model": MODEL,
                    "prompt": prompt,
                    "stream": False,
                    "options": {
                        "temperature": 0
                    }
                },
                timeout=600
            )

            result = response.json() # extract LLM response from JSON object
            print(type(result))
            print(type(response))
            print(response)
            
            print(type(response.elapsed))
            print(response.elapsed)
            return result
        except Exception as e:
            # Log error and return None so status can be marked 'failed'
            print(e)
            return None

    def resume_loop(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        # TODO
        # 1. Query: SELECT * FROM data WHERE status != 'completed'
        # 2. Gather tasks for send_single_prompt
        # 3. Use Transactional Commits (BEGIN/COMMIT) for batch writes

        pending_rows = self.con.execute("""
            SELECT ID, prompt 
            FROM prompts 
            WHERE status = 'Pending'
        """).fetchall()

        print(pending_rows)
        for row in pending_rows:
            ID, prompt_text = row
            print(prompt_text)

            try:
                response = self.send_single_prompt(row_id = ID, prompt = prompt_text) 
                
                predicted_pathogenicty = Pathogenicity(response.get('response'))
                
                # --- UPDATE STATUS ---
                self.con.execute("""
                    UPDATE prompts 
                    SET 
                        ---status = 'Completed', 
                        response = ?, 
                        prediction = ?
                    WHERE ID = ?
                """, (response, predicted_pathogenicity, ID))
                
                print(f"Processed {ID}: {predicted_pathogenicity}")

            except Exception as e:
                print(f"Error on {ID}: {e}")
                # Mark as ERROR so we don't just crash and lose track
                self.con.execute("""UPDATE prompts SET status = 'Wrong' WHERE "ID" = ?""", (ID,))

        print("Job done.")
        self.con.close()    
        
        pass

    def finalize(self):
        """
        This is a reST style.
        
        :param param1: this is a first param
        :param param2: this is a second param
        :returns: this is a description of what is returned
        :raises keyError: raises an exception
        """
        pass

In [6]:
# --- Test Cases ---

# Case 1: The "No Spaces" / CamelCase LLM Hallucination
print(Pathogenicity(classification="LikelyPathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 2: The "Forgot Spaces" inside a Paragraph
print(Pathogenicity(classification="Review suggests this is LikelyBenign based on frequency."))
# Output: classification=<Classification.LIKELY_BENIGN: 'likelybenign'>

# Case 3: Standard Spacing
print(Pathogenicity(classification="Likely Pathogenic"))
# Output: classification=<Classification.LIKELY_PATHOGENIC: 'likelypathogenic'>

# Case 4: Long Squashed Phrase
print(Pathogenicity(classification="variantofuncertainsignificance"))
# Output: classification=<Classification.VUS: 'vus'>

print(Pathogenicity(classification="Variant of uncertain significance"))

print(Pathogenicity(classification="Classification: 'Pathogenic'"))

{<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>}
classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
{<Classification.LIKELY_BENIGN: 'Likely Benign'>}
classification=<Classification.LIKELY_BENIGN: 'Likely Benign'>
{<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>}
classification=<Classification.LIKELY_PATHOGENIC: 'Likely Pathogenic'>
{<Classification.VUS: 'VUS'>}
classification=<Classification.VUS: 'VUS'>
{<Classification.VUS: 'VUS'>}
classification=<Classification.VUS: 'VUS'>
{<Classification.PATHOGENIC: 'Pathogenic'>}
classification=<Classification.PATHOGENIC: 'Pathogenic'>


In [ ]:
print(Pathogenicity(classification="Pathogenic Pathogenic"))
print(Pathogenicity(classification="PathogenicPathogenic"))
print(Pathogenicity(classification="Im not sure but its probably not benign but pathogenic benign'"))

In [5]:
p = Prompter(db_path='prompts.db')

BinderException: Binder Error: DEFAULT value cannot contain column names

LINE 1: ... TABLE prompts ADD COLUMN IF NOT EXISTS status STRING DEFAULT "PENDING";
                                                                         ^

In [31]:
print(p.send_single_prompt(row_id = 'test', prompt = prompt_test))

<class 'dict'>
<class 'requests.models.Response'>
<Response [200]>
<class 'datetime.timedelta'>
0:00:53.441248
{'model': 'tinyllama', 'created_at': '2026-02-13T18:44:47.982604882Z', 'response': "Yes, I am a helpful AI assistant. Here's an example of how I can assist you:\n\nDear [Name],\n\nAs a valued customer of our company, we would like to introduce ourselves and provide you with some information about our products and services. We are excited to share our latest innovations and offerings with you.\n\nOur company is committed to providing high-quality products and exceptional customer service. Our team of experts is dedicated to ensuring that all customers receive the best possible experience.\n\nHere's a brief overview of some of our most popular products:\n\n1. [Product 1]: This product is designed to solve [problem] for [target audience]. It features [unique selling point], which sets it apart from competitors.\n\n2. [Product 2]: This product is perfect for [specific use case]. I

In [14]:
p.resume_loop()

[]
Job done.


In [37]:
# This prints the schema as a table
print(p.con.execute("PRAGMA table_info('prompts')").pl())

shape: (36, 6)
┌─────┬─────────────────┬─────────┬─────────┬────────────┬───────┐
│ cid ┆ name            ┆ type    ┆ notnull ┆ dflt_value ┆ pk    │
│ --- ┆ ---             ┆ ---     ┆ ---     ┆ ---        ┆ ---   │
│ i32 ┆ str             ┆ str     ┆ bool    ┆ str        ┆ bool  │
╞═════╪═════════════════╪═════════╪═════════╪════════════╪═══════╡
│ 0   ┆ model           ┆ VARCHAR ┆ false   ┆ null       ┆ false │
│ 1   ┆ prompt_template ┆ VARCHAR ┆ false   ┆ null       ┆ false │
│ 2   ┆ temp            ┆ DOUBLE  ┆ false   ┆ null       ┆ false │
│ 3   ┆ repeat          ┆ BIGINT  ┆ false   ┆ null       ┆ false │
│ 4   ┆ #ID             ┆ BIGINT  ┆ false   ┆ null       ┆ false │
│ …   ┆ …               ┆ …       ┆ …       ┆ …          ┆ …     │
│ 31  ┆ comment         ┆ VARCHAR ┆ false   ┆ null       ┆ false │
│ 32  ┆ variant_id      ┆ VARCHAR ┆ false   ┆ null       ┆ false │
│ 33  ┆ prompt          ┆ VARCHAR ┆ false   ┆ null       ┆ false │
│ 34  ┆ status          ┆ VARCHAR ┆ false   ┆ '